In [1]:
import numpy as np
from PIL import Image as PILImage, ImageDraw
from enum import Enum
import random
import gymnasium as gym


# Definitions
class LightColor(Enum):
    RED = 0
    YELLOW = 1
    GREEN = 2


class Cell(Enum):
    ROAD = 0
    HOUSE = 1
    TRAFFIC = 2
    HOSPITAL = 3
    CAR = 4


class GridAction(Enum):
    LEFT = 0
    DOWN = 1
    RIGHT = 2
    UP = 3


# Environment
class SmartAmbulanceEnv(gym.Env):
    """
    Custom Gymnasium Environment for a Smart Ambulance pathfinding task.

    Attributes:
        size (int): Grid dimensions (size x size).
        _grid (numpy.ndarray): 2D array representing the map layout.
        traffic_lights (dict): Stores (r, c) positions and current [LightColor, timer].
        car_positions (set): Current coordinates of dynamic car obstacles.
        car_move_rate (int): Frequency of car movements (every N steps).
    """

    def __init__(self, size=20, seed=42):
        self.size = size
        self.rows = size
        self.cols = size
        self._state = 0

        random.seed(seed)
        np.random.seed(seed)

        self._grid = np.zeros((size, size), dtype=int)
        self.car_move_counter = 0
        self.car_move_rate = 3  # Every how many steps cars move

        # Houses
        for _ in range(20):
            r, c = random.randint(0, size - 1), random.randint(0, size - 1)
            if (r, c) != (0, 0) and (r, c) != (size - 1, size - 1):
                self._grid[r, c] = Cell.HOUSE.value

        # Moving Cars
        for _ in range(10):
            r, c = random.randint(0, size - 1), random.randint(0, size - 1)
            if self._grid[r, c] == Cell.ROAD.value and (r, c) != (0, 0):
                self._grid[r, c] = Cell.CAR.value

        # Save Car Positions
        self.car_positions = set()
        for r in range(size):
            for c in range(size):
                if self._grid[r, c] == Cell.CAR.value:
                    self.car_positions.add((r, c))

        # Traffic Lights
        self.traffic_lights = {}
        for _ in range(15):
            r, c = random.randint(0, size - 1), random.randint(0, size - 1)
            if self._grid[r, c] == Cell.ROAD.value and (r, c) != (0, 0):
                self._grid[r, c] = Cell.TRAFFIC.value
                self.traffic_lights[(r, c)] = [random.choice(list(LightColor)), 0]

        self._grid[size - 1, size - 1] = Cell.HOSPITAL.value

        self.observation_space = gym.spaces.Discrete(size * size)
        self.action_space = gym.spaces.Discrete(len(GridAction))

    def update_lights(self):
        """Updates the state and timers of all traffic lights in the grid."""
        for loc in self.traffic_lights:
            color, counter = self.traffic_lights[loc]
            counter += 1
            limit = (
                10
                if color == LightColor.GREEN
                else 5 if color == LightColor.YELLOW else 10
            )
            if counter >= limit:
                next_c = (
                    LightColor.YELLOW
                    if color == LightColor.GREEN
                    else (
                        LightColor.RED
                        if color == LightColor.YELLOW
                        else LightColor.GREEN
                    )
                )
                self.traffic_lights[loc] = [next_c, 0]
            else:
                self.traffic_lights[loc] = [color, counter]

    def move_cars(self):
        """
        Handles dynamic obstacle movement.
        Cars move randomly to adjacent road cells while avoiding houses,
        hospitals, traffic lights, and other cars.
        """
        self.car_move_counter += 1

        if self.car_move_counter % self.car_move_rate != 0:
            return
        BLOCKED = {Cell.HOUSE.value, Cell.HOSPITAL.value, Cell.TRAFFIC.value}
        directions = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        new_positions = set()
        amb_r, amb_c = divmod(self._state, self.cols)

        for r, c in list(self.car_positions):
            shuffled = directions[:]
            random.shuffle(shuffled)
            moved = False
            for dr, dc in shuffled:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < self.rows and 0 <= nc < self.cols):
                    continue
                if self._grid[nr, nc] in BLOCKED:
                    continue
                if (nr, nc) in new_positions:
                    continue
                if (nr, nc) == (amb_r, amb_c):
                    continue

                self._grid[r, c] = Cell.ROAD.value
                self._grid[nr, nc] = Cell.CAR.value
                new_positions.add((nr, nc))
                moved = True
                break
            if not moved:
                new_positions.add((r, c))

        self.car_positions = new_positions

    def reset(self, seed=None, options=None):
        """Resets the ambulance to the start (0,0) and randomizes light phases."""
        self._state = 0
        for loc in self.traffic_lights:
            self.traffic_lights[loc] = [random.choice(list(LightColor)), 0]
        return self._state, {}

    def step(self, action):
        """
        Executes one time step in the environment.
        Args:
            action (int): Direction to move (0:Left, 1:Down, 2:Right, 3:Up).
        Returns:
            tuple: (new_state, reward, done, truncated, info)
        """
        self.update_lights()
        self.move_cars()

        r, c = divmod(self._state, self.cols)
        moves = {
            GridAction.LEFT.value: (0, -1),
            GridAction.DOWN.value: (1, 0),
            GridAction.RIGHT.value: (0, 1),
            GridAction.UP.value: (-1, 0),
        }
        dr, dc = moves[action]
        nr = max(0, min(r + dr, self.rows - 1))
        nc = max(0, min(c + dc, self.cols - 1))

        cell = self._grid[nr, nc]
        new_state = self._state
        reward = -1
        done = False

        if cell == Cell.TRAFFIC.value:
            color = self.traffic_lights[(nr, nc)][0]
            if color == LightColor.RED:
                reward = -10
            elif color == LightColor.YELLOW:
                reward = -3
                new_state = nr * self.cols + nc
            else:
                reward = -1
                new_state = nr * self.cols + nc
        elif cell == Cell.HOUSE.value:
            reward = -20
        elif cell == Cell.CAR.value:
            reward = -15
        else:
            new_state = nr * self.cols + nc
            if cell == Cell.HOSPITAL.value:
                reward = 100
                done = True
            else:
                reward = -1

        prev_state = self._state
        self._state = new_state
        # check if new state is same as previous
        if new_state == prev_state:
            reward = -5

        return new_state, reward, done, False, {}

    def render_frame(self, state, lights_snapshot):
        """
        Generates an RGB image representing the current grid state.
        Args:
            state (int): Current ambulance position.
            lights_snapshot (dict): Current state of traffic lights for rendering.
        Returns:
            numpy.ndarray: The rendered frame.
        """
        cell_size = 40
        img = PILImage.new(
            "RGB", (self.cols * cell_size, self.rows * cell_size), "#808080"
        )
        draw = ImageDraw.Draw(img)

        for r in range(self.rows):
            for c in range(self.cols):
                rect = [
                    c * cell_size,
                    r * cell_size,
                    (c + 1) * cell_size,
                    (r + 1) * cell_size,
                ]
                draw.rectangle(rect, outline="black", width=1)
                val = self._grid[r, c]

                # House
                if val == Cell.HOUSE.value:
                    x1 = c * cell_size + 5
                    y1 = r * cell_size + 10
                    x2 = (c + 1) * cell_size - 5
                    y2 = (r + 1) * cell_size - 5
                    draw.rectangle([x1, y1, x2, y2], fill="#8B4513", outline="black")
                    draw.polygon(
                        [(x1, y1), ((x1 + x2) // 2, y1 - 10), (x2, y1)], fill="#A52A2A"
                    )
                    draw.rectangle([x1 + 8, y2 - 12, x1 + 16, y2], fill="#654321")
                    draw.rectangle([x2 - 15, y1 + 5, x2 - 5, y1 + 15], fill="#ADD8E6")

                # Hospital
                elif val == Cell.HOSPITAL.value:
                    draw.rectangle(rect, fill="white", outline="black")
                    cx = c * cell_size + cell_size // 2
                    cy = r * cell_size + cell_size // 2
                    draw.line([cx, cy - 10, cx, cy + 10], fill="red", width=4)
                    draw.line([cx - 10, cy, cx + 10, cy], fill="red", width=4)

                # Car
                elif val == Cell.CAR.value:
                    x1 = c * cell_size + 6
                    y1 = r * cell_size + 14
                    x2 = (c + 1) * cell_size - 6
                    y2 = (r + 1) * cell_size - 8
                    draw.rounded_rectangle(
                        [x1, y1, x2, y2], radius=6, fill="#3498db", outline="black"
                    )
                    draw.rectangle([x1 + 6, y1 - 6, x2 - 6, y1 + 4], fill="#2980b9")
                    draw.rectangle([x1 + 8, y1 - 4, x2 - 8, y1 + 2], fill="#add8e6")
                    draw.ellipse([x1 + 2, y2 - 6, x1 + 8, y2], fill="black")
                    draw.ellipse([x2 - 8, y2 - 6, x2 - 2, y2], fill="black")

                # Traffic Light
                elif val == Cell.TRAFFIC.value:
                    color_obj = lights_snapshot.get((r, c))[0]
                    c_hex = (
                        "#27ae60"
                        if color_obj == LightColor.GREEN
                        else "#f1c40f" if color_obj == LightColor.YELLOW else "#e74c3c"
                    )
                    draw.ellipse(
                        [
                            c * cell_size + 10,
                            r * cell_size + 10,
                            (c + 1) * cell_size - 10,
                            (r + 1) * cell_size - 10,
                        ],
                        fill=c_hex,
                        outline="black",
                    )

        # Ambulance
        r_curr, c_curr = divmod(state, self.cols)
        x1 = c_curr * cell_size + 8
        y1 = r_curr * cell_size + 12
        x2 = (c_curr + 1) * cell_size - 8
        y2 = (r_curr + 1) * cell_size - 12
        draw.rounded_rectangle(
            [x1, y1, x2, y2], radius=6, fill="#f1c40f", outline="black", width=2
        )
        draw.rectangle([x1 + 6, y1 + 4, x2 - 6, y1 + 10], fill="#add8e6")
        draw.ellipse([x1 + 2, y2 - 6, x1 + 8, y2], fill="black")
        draw.ellipse([x2 - 8, y2 - 6, x2 - 2, y2], fill="black")

        return np.array(img)


# DEVELOPER DOCUMENTATION: ALGORITHM IMPLEMENTATION GUIDE
"""
GUIDE FOR IMPLEMENTING ALGORITHMS RL:

1. ENVIRONMENT INTERFACE:
   - Reset the environment using `state, info = env.reset()`.
   - Take actions using `next_state, reward, done, truncated, info = env.step(action)`.
   - Action Mapping: {0: Left, 1: Down, 2: Right, 3: Up}.

2. STATE SPACE:
   - The state is a Discrete integer from 0 to (size*size - 1).
   - Coordinate Conversion: `row, col = divmod(state, env.cols)`.

3. REWARD SYSTEM & CONSTRAINTS:
   - Default step/Green light: -1
   - Yellow Light: -3 (Ambulance can pass but penalized)
   - Red Light: -10 (Ambulance stays in place if it attempts to enter)
   - Collision (House): -20 (Ambulance stays in place)
   - Collision (Moving Car): -15 (Ambulance stays in place)
   - Returning to the same state: -5
   - Goal (Hospital): +100 (Episode ends)

4. DYNAMIC OBSTACLES:
   - Dynamic cars move every `env.car_move_rate` steps.
   - Traffic lights cycle through Green -> Yellow -> Red automatically.

5. RENDERING FOR ANIMATION:
   - Required imports:
       import matplotlib.pyplot as plt
       from matplotlib.animation import FuncAnimation
       import copy
   - Store states and traffic light snapshots during the episode:
       - Save `state` at each step.
       - Save a deep copy of traffic lights using:
           copy.deepcopy(env.traffic_lights)
   - Use:
       env.render_frame(state, light_snapshot)
     to generate frames.
   - Create animation using Matplotlib:
       - Initialize a figure and axis using plt.subplots()
       - Use FuncAnimation to update frames over time
       - Display using plt.show()
   - Notes:
       - copy.deepcopy is important to avoid overwriting past light states.
       - This method works for visualization only and is independent of training.
   - Optional (for Tensor-based approaches):
       - You can convert frames to arrays using NumPy (already returned).
       - These frames can be stored and used later for video generation or deep learning models.
"""

'\nGUIDE FOR IMPLEMENTING ALGORITHMS RL:\n\n1. ENVIRONMENT INTERFACE:\n   - Reset the environment using `state, info = env.reset()`.\n   - Take actions using `next_state, reward, done, truncated, info = env.step(action)`.\n   - Action Mapping: {0: Left, 1: Down, 2: Right, 3: Up}.\n\n2. STATE SPACE:\n   - The state is a Discrete integer from 0 to (size*size - 1).\n   - Coordinate Conversion: `row, col = divmod(state, env.cols)`.\n\n3. REWARD SYSTEM & CONSTRAINTS:\n   - Default step/Green light: -1\n   - Yellow Light: -3 (Ambulance can pass but penalized)\n   - Red Light: -10 (Ambulance stays in place if it attempts to enter)\n   - Collision (House): -20 (Ambulance stays in place)\n   - Collision (Moving Car): -15 (Ambulance stays in place)\n   - Returning to the same state: -5\n   - Goal (Hospital): +100 (Episode ends)\n\n4. DYNAMIC OBSTACLES:\n   - Dynamic cars move every `env.car_move_rate` steps.\n   - Traffic lights cycle through Green -> Yellow -> Red automatically.\n\n5. RENDERING

#Algorithms

#SARSA

In [2]:
import numpy as np
import random

#create environment
env = SmartAmbulanceEnv(size=20)

n_states = env.size * env.size
n_actions = 4

Q = np.zeros((n_states, n_actions))
# Hyperparameters
alpha = 0.1
gamma = 0.97
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
episodes = 3000

#epsilon-greedy policy
def choose_action(state):
    if random.random() < epsilon:
        return random.randint(0, 3)   # exploration
    else:
        return np.argmax(Q[state])    # exploitation
#Trining
episode_rewards = []

for episode in range(episodes):

    state, _ = env.reset()
    action = choose_action(state)

    done = False
    total_reward = 0

    while not done:

        next_state, reward, done, _, _ = env.step(action)

        next_action = choose_action(next_state)

        # SARSA update rule
        Q[state, action] = Q[state, action] + alpha * (
            reward +
            gamma * Q[next_state, next_action] -
            Q[state, action]
        )

        state = next_state
        action = next_action

        total_reward += reward

    episode_rewards.append(total_reward)

    # decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode} | Reward: {total_reward} | Epsilon: {epsilon:.3f}")

Episode 0 | Reward: -17045 | Epsilon: 0.995
Episode 10 | Reward: -871 | Epsilon: 0.946
Episode 20 | Reward: -3673 | Epsilon: 0.900
Episode 30 | Reward: -764 | Epsilon: 0.856
Episode 40 | Reward: -816 | Epsilon: 0.814
Episode 50 | Reward: -284 | Epsilon: 0.774
Episode 60 | Reward: -67 | Epsilon: 0.737
Episode 70 | Reward: -229 | Epsilon: 0.701
Episode 80 | Reward: -294 | Epsilon: 0.666
Episode 90 | Reward: -30 | Epsilon: 0.634
Episode 100 | Reward: -494 | Epsilon: 0.603
Episode 110 | Reward: -259 | Epsilon: 0.573
Episode 120 | Reward: -135 | Epsilon: 0.545
Episode 130 | Reward: -124 | Epsilon: 0.519
Episode 140 | Reward: -134 | Epsilon: 0.493
Episode 150 | Reward: -185 | Epsilon: 0.469
Episode 160 | Reward: -135 | Epsilon: 0.446
Episode 170 | Reward: -5 | Epsilon: 0.424
Episode 180 | Reward: -23 | Epsilon: 0.404
Episode 190 | Reward: -46 | Epsilon: 0.384
Episode 200 | Reward: -58 | Epsilon: 0.365
Episode 210 | Reward: -148 | Epsilon: 0.347
Episode 220 | Reward: -85 | Epsilon: 0.330
Epis

Testing / Evaluation

In [3]:
state, _ = env.reset()
done = False
total_reward = 0

while not done:
    action = np.argmax(Q[state])
    state, reward, done, _, _ = env.step(action)
    total_reward += reward
    print(state, reward)

print("Final Reward:", total_reward)

1 -1
21 -1
41 -1
42 -1
62 -1
82 -1
102 -1
102 -5
102 -5
122 -1
123 -1
124 -1
125 -1
126 -1
146 -1
147 -1
148 -1
168 -1
169 -1
149 -1
150 -1
151 -1
171 -1
191 -1
211 -1
212 -1
212 -5
213 -1
214 -1
234 -1
254 -1
274 -1
294 -1
314 -1
315 -1
316 -1
336 -1
356 -1
357 -1
377 -1
378 -1
378 -5
398 -1
399 100
Final Reward: 41


In [4]:
# mean
window = 100
means = []

for i in range(0, len(episode_rewards), window):
    means.append(np.mean(episode_rewards[i:i+window]))

print(means)

[np.float64(-1297.67), np.float64(-181.72), np.float64(-91.0), np.float64(-60.82), np.float64(-32.27), np.float64(-12.97), np.float64(-13.1), np.float64(9.41), np.float64(17.99), np.float64(39.47), np.float64(38.25), np.float64(42.37), np.float64(43.66), np.float64(48.37), np.float64(50.04), np.float64(51.32), np.float64(41.56), np.float64(39.4), np.float64(46.8), np.float64(45.11), np.float64(47.66), np.float64(46.78), np.float64(44.93), np.float64(47.71), np.float64(47.06), np.float64(44.15), np.float64(43.96), np.float64(43.87), np.float64(47.68), np.float64(44.48)]


comparison between actual path and shortest path

In [5]:
def manhattan_distance(start, goal, size):
    sx, sy = start // size, start % size
    gx, gy = goal // size, goal % size
    return abs(sx - gx) + abs(sy - gy)

In [7]:
path = []

state, _ = env.reset()
done = False

while not done:
    action = np.argmax(Q[state])
    next_state, reward, done, _, _ = env.step(action)

    path.append(state)
    state = next_state

In [8]:
start = path[0]
goal = path[-1]

shortest = manhattan_distance(start, goal, env.size)

print("Actual steps:", len(path))
print("Theoretical shortest:", shortest + 1)

Actual steps: 40
Theoretical shortest: 38


# Comparison Before and After

In [9]:
def evaluate_policy(env, Q, episodes=50):
    rewards = []

    for _ in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = np.argmax(Q[state])   # exploitation only
            state, reward, done, _, _ = env.step(action)
            total_reward += reward

        rewards.append(total_reward)

    print("Average Reward:", np.mean(rewards))
    return np.mean(rewards)

In [10]:
def run_random(env, episodes=50):
    rewards = []

    for _ in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = random.randint(0, 3)
            state, reward, done, _, _ = env.step(action)
            total_reward += reward

        rewards.append(total_reward)

    return np.mean(rewards)

In [11]:
random_score = run_random(env)
sarsa_score = evaluate_policy(env, Q)

print("Random Policy:", random_score)
print("SARSA Policy:", sarsa_score)

Average Reward: 44.6
Random Policy: -4870.12
SARSA Policy: 44.6
